In [5]:
pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

import warnings

# Suppress harmless warnings for clean execution logs
warnings.filterwarnings('ignore')

### 1. DATA INGESTION & STANDARDIZATION

In [7]:
# Load data
df = pd.read_csv("NexaFlow_SaaS_Funnel_Optimization/01_Data/raw/saas_funnel_raw.csv")

In [8]:
# Standardize column naming conventions for MySQL (snake_case)
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [9]:
# Drop exact duplicates
df = df.drop_duplicates()

In [10]:
# Standardize categorical string columns (Title Case, stripped of whitespace)
cat_cols = ['acquisition_channel', 'country', 'plan_type']

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    # Replace 'Nan' string back to actual NaN for proper missing value handling
    df[col] = df[col].replace('Nan', pd.NA)

### 2. DATA TYPE CASTING

In [11]:
# Parse standard datetime columns
standard_date_cols = ['signup_date', 'activation_date', 'subscription_date']

for col in standard_date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [12]:
# Parse nanosecond epoch timestamps
df['churn_date'] = pd.to_datetime(df['churn_date'], unit='ns', errors='coerce')

In [13]:
# Cast counts and metrics to appropriate numeric types
# Utilizing pandas Nullable Integer type ('Int64') where NaN values exist
df['sessions_count'] = df['sessions_count'].clip(lower=0, upper=50).astype('Int64')
df['actions_count'] = np.floor(df['actions_count']).clip(lower=0).astype('Int64')

### 3. FUNNEL LOGIC & TIMELINE ENFORCEMENT

In [14]:
# Rule A: Downstream funnel completion mandates upstream completion

df.loc[df['is_converted'] == 1, 'is_activated'] = 1

In [15]:
# Rule B: Time Consistency Checks (Chronological integrity)
# If activation is before signup, it's invalid data. Wipe the activation.
df.loc[df['activation_date'] < df['signup_date'], 'activation_date'] = pd.NaT

# If subscription is before signup, it's invalid.
df.loc[df['subscription_date'] < df['signup_date'], 'subscription_date'] = pd.NaT

# If churn is before subscription (for converted) or signup (for non-converted)
df.loc[df['churn_date'] < df['subscription_date'], 'churn_date'] = pd.NaT
df.loc[df['churn_date'] < df['signup_date'], 'churn_date'] = pd.NaT

In [16]:
# Rule C: Synchronize flags with valid dates
# If dates were wiped due to logical errors, the boolean flags must reflect the drop-off
df.loc[df['activation_date'].isna(), 'is_activated'] = 0

df.loc[df['subscription_date'].isna(), 'is_converted'] = 0

df.loc[df['churn_date'].isna(), 'is_churned'] = 0

In [17]:
# Rule D: Clean dependent columns based on finalized flags
# Non-activated users
df.loc[df['is_activated'] == 0, ['activation_date', 'days_to_activate']] = [pd.NaT, pd.NA]

In [18]:
# Non-converted users
df.loc[df['is_converted'] == 0, ['subscription_date', 'days_to_convert', 'plan_type']] = [pd.NaT, pd.NA, 'None']
df.loc[df['is_converted'] == 0, 'revenue'] = 0

# Non-churned users
df.loc[df['is_churned'] == 0, 'churn_date'] = pd.NaT

### 4. FEATURE ENGINEERING (SQL & Power BI Optimized)

In [19]:
# Recalculate duration metrics dynamically based on validated dates
df['days_to_activate'] = (df['activation_date'] - df['signup_date']).dt.days.astype('Int64')
df['days_to_convert'] = (df['subscription_date'] - df['signup_date']).dt.days.astype('Int64')

In [20]:
# Active Days Calculation (Time to churn, or time to max observation date)
df["time_to_churn"] = (df["churn_date"] - df["signup_date"]).dt.days

max_observation_date = df['signup_date'].max()
df['active_days'] = np.where(
    df['is_churned'] == 1,
    (df['churn_date'] - df['signup_date']).dt.days,
    (max_observation_date - df['signup_date']).dt.days
)

df['active_days'] = pd.to_numeric(df['active_days']).astype('Int64')

In [21]:
# Cohort Month: Truncated to the first of the month for SQL/Power BI date tables
df['cohort_month'] = df['signup_date'].dt.to_period('M').dt.to_timestamp()

In [22]:
# Engagement Segments
df['engagement_segment'] = pd.qcut(
    df['sessions_count'].rank(method='first'), 
    q=3, 
    labels=['Low', 'Medium', 'High']
)

In [23]:
# High Value Channel Flag
conversion_rates = df.groupby('acquisition_channel')['is_converted'].transform('mean')
overall_avg_conversion = df['is_converted'].mean()
df['is_high_value_channel'] = (conversion_rates > overall_avg_conversion).astype(int)

### 5. FINAL VALIDATION & EXPORT

In [24]:
# Sort by user_id for clean database insertion
df = df.sort_values('user_id').reset_index(drop=True)

In [25]:
USER = 'root'
PASSWORD = 'nyc%4022182021'
HOST = 'localhost'
PORT = 3306
DATABASE = 'saas_funnel_db'
TABLE_NAME = 'saas_funnel'

# Create SQLAlchemy engine for MySQL
conn = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(conn)

try:
    # Write DataFrame to MySQL
    df.to_sql(
        name=TABLE_NAME,
        con=engine,
        if_exists='replace',
        index=False
    )
    print(f"DataFrame successfully written to MySQL table '{TABLE_NAME}'.")
except SQLAlchemyError as e:
    print("Error while writing to MySQL:", e)
finally:
    engine.dispose()  # Close connection

DataFrame successfully written to MySQL table 'saas_funnel'.
